# AVA-VLA Suite-balanced K4 · 본 학습 1,000 step

완료된 파일럿의 Drive 인계 결과를 사용합니다. 같은 실행 그룹은 한 런타임에서 실행하세요. 새 Colab A100 런타임에서 `HF_TOKEN` 접근을 허용하고 1~3번 셀을 실행하세요. 4번의 **RUN_MAIN**을 켠 뒤 실행하면 본 학습이 시작됩니다.

- 체크포인트는 optimizer step **250·500·750·1,000**에만 저장합니다. 검증은 기존 100-step 주기를 유지합니다. 1,000-step 모델은 PC 보관용 최종 체크포인트입니다.
- 파일럿 보고서와 체크포인트는 Drive에서 검증하여 복원합니다. 소스는 파일럿과 같은 GitHub 기준 커밋과 동일한 로컬 수정 커밋으로 재현합니다. 학습 입력은 기존 HF 고정 자산을 사용합니다.
- 첫 실행은 T0에서 시작합니다. 같은 실행 그룹에 봉인된 체크포인트가 있으면 가장 최근 step에서 자동으로 이어가고, 1,000-step 체크포인트가 있으면 상태만 확인합니다. 250 전에 중단되면 T0에서 다시 시작합니다. 시간 예산 종료 시에도 지정 step만 저장합니다. 재개 실행에서는 상태를 읽으면 `RESUME_STATE_OK`, 첫 optimizer update를 실제 마치면 `RESUME_FIRST_STEP_OK`가 표시됩니다.
- 각 체크포인트를 Drive에 복사해 검증합니다. 750 step의 Drive 검증 뒤 250 step의 Colab SSD 사본을, 1,000 step의 Drive 검증 뒤 500 step의 Colab SSD 사본을 삭제합니다. 두 경우 모두 삭제할 체크포인트의 Drive 사본도 검증합니다. Drive에는 체크포인트 4개 기준 약 **122.9 GB**가 필요합니다. 로컬 SSD에는 새 체크포인트 작성 공간과 파일럿 복원본·입력·런타임 공간도 필요합니다.
- Hugging Face는 기존 고정 입력을 읽는 데 사용합니다. 본 학습의 자동 업로드는 꺼져 있습니다. 학습이 끝나면 아래 Windows 안내에 따라 최종 체크포인트를 PC에 복사할 수 있습니다. Gmail과 완료 후 런타임 해제는 4번 셀에서 선택할 수 있습니다.


In [ ]:
# @title 1. 완료된 파일럿 선택 · Drive 연결
from pathlib import Path
import json
import os

from google.colab import drive, userdata

PILOT_RUN_GROUP = "fish13031e-ava-suite-k4-seed42-20261111T062518Z-2f8a601c" # @param {type:"string"}
drive.mount("/content/drive")
DRIVE_BASE = Path("/content/drive/MyDrive/VLADE/runs/ava_fft_k4")
if not PILOT_RUN_GROUP or Path(PILOT_RUN_GROUP).name != PILOT_RUN_GROUP:
    raise ValueError("PILOT_RUN_GROUP에 완료된 파일럿 실행 그룹을 넣으세요.")
DRIVE_RUN = DRIVE_BASE / PILOT_RUN_GROUP
HANDOFF = DRIVE_RUN / "pilot_handoff"
DRIVE_OUTPUT = DRIVE_RUN / "main_fft"
meta = json.loads((HANDOFF / "handoff.json").read_text(encoding="utf-8"))
if (
    meta["schema_version"] != 1
    or meta["run_group"] != PILOT_RUN_GROUP
    or meta["workspace"] != "/content/vlade_work"
    or meta["repository"] != "https://github.com/eastha10/VLADE.git"
    or meta["config"] != "configs/research/ava_suite.json"
):
    raise RuntimeError("기존 Suite K4 파일럿 인계 정보가 필요합니다.")
token = userdata.get("HF_TOKEN")
if not token:
    raise RuntimeError("Colab Secrets의 HF_TOKEN 접근을 허용하세요.")
os.environ["HF_TOKEN"] = token
del token
print("PILOT_RUN_GROUP:", PILOT_RUN_GROUP)
print("파일럿 Drive 경로:", HANDOFF)
print("본 학습 Drive 경로:", DRIVE_OUTPUT)


In [ ]:
# @title 2. 파일럿과 동일한 소스 재현 · 체크포인트 복원
import os
import re
import subprocess
import tempfile
from pathlib import Path

SOURCE_REF = "afb670a2c9bb3befe03a01c6ac2eaf3c99aca769"
LEGACY_SOURCE = "afb670a2c9bb3befe03a01c6ac2eaf3c99aca769"
REPOSITORY = "https://github.com/eastha10/VLADE.git"
OLD_LOCAL_COMMIT = "5bb7be5b266e70df69f677b651fe4acea4a9b33e"
SOURCE_PATCH_VERSION = 2
PATCH_PATH = "src/vlade/runtime/ava_patch.py"
OLD_SHA = "0a9c25300f3358f87af1719199608af3c27567dba0425690327fccd44cfb516b"
LF_SHA = "aeca129d863bec439623f54d93810792d82c86921c25fbe69d9169ffdb373788"

LAYER_PATH = "src/vlade/models/teachers/backbones/prismatic.py"


def source_git(code, *args, env=None):
    return subprocess.check_output(
        ["git", "-C", str(code), *args], text=True, env=env
    ).strip()


def patch_legacy_source(code):
    head = source_git(code, "rev-parse", "HEAD")
    if head not in {LEGACY_SOURCE, OLD_LOCAL_COMMIT}:
        raise RuntimeError("예상한 VLADE 소스 커밋을 찾지 못했습니다.")
    if source_git(code, "status", "--porcelain", "--ignore-submodules=all"):
        raise RuntimeError("깨끗한 VLADE 소스 checkout이 필요합니다.")

    patch_module = code / PATCH_PATH
    payload = patch_module.read_bytes()
    old = f'INSTALLED_SHA = "{OLD_SHA}"'.encode()
    fixed = f'INSTALLED_SHA = "{LF_SHA}"'.encode()
    expected = old if head == LEGACY_SOURCE else fixed
    if payload.count(expected) != 1:
        raise RuntimeError("예상한 AVA 설치 해시 정의를 찾지 못했습니다.")

    layer_module = code / LAYER_PATH
    layer_source = layer_module.read_bytes()
    old_binding = b"_ls_new_forward.__get__(ls_module, LayerScale)"
    new_binding = b"_ls_new_forward.__get__(ls_module, type(ls_module))"
    if layer_source.count(old_binding) != 1:
        raise RuntimeError("예상한 LayerScale forward 참조를 찾지 못했습니다.")

    if head == LEGACY_SOURCE:
        patch_module.write_bytes(payload.replace(old, fixed))
    layer_module.write_bytes(layer_source.replace(old_binding, new_binding))
    source_git(code, "add", "--", PATCH_PATH, LAYER_PATH)
    # Fixed metadata and the same parent produce the same commit on every runtime.
    identity = {
        "GIT_AUTHOR_NAME": "VLADE Colab pilot",
        "GIT_AUTHOR_EMAIL": "vlade-pilot@localhost",
        "GIT_COMMITTER_NAME": "VLADE Colab pilot",
        "GIT_COMMITTER_EMAIL": "vlade-pilot@localhost",
        "GIT_AUTHOR_DATE": "2026-11-11T00:00:00Z",
        "GIT_COMMITTER_DATE": "2026-11-11T00:00:00Z",
    }
    commit = source_git(
        code, "-c", "commit.gpgsign=false", "commit-tree",
        source_git(code, "write-tree"), "-p", LEGACY_SOURCE,
        "-m", "Fix AVA patch LF hash and LayerScale method binding",
        env={**os.environ, **identity},
    )
    source_git(code, "checkout", "--quiet", "--detach", commit)
    return commit


def prepare_source(ref, source_root=Path("/content/vlade_sources")):
    if not re.fullmatch(r"[0-9a-f]{40}", ref):
        raise ValueError("SOURCE_REF에 GitHub의 40자리 커밋 SHA를 지정하세요.")
    source_root.mkdir(parents=True, exist_ok=True)
    code = Path(tempfile.mkdtemp(prefix=f"ava-{ref[:8]}-", dir=source_root))
    source_git(code, "init", "--quiet")
    source_git(code, "fetch", "--quiet", "--depth=1", REPOSITORY, ref)
    source_git(code, "checkout", "--quiet", "--detach", "FETCH_HEAD")
    if source_git(code, "rev-parse", "HEAD") != ref:
        raise RuntimeError("고정 Git 커밋이 일치하지 않습니다.")
    if ref == LEGACY_SOURCE:
        return code, patch_legacy_source(code)

    fixed = f'INSTALLED_SHA = "{LF_SHA}"'.encode()
    if (code / PATCH_PATH).read_bytes().count(fixed) != 1:
        raise RuntimeError("이 커밋에는 확인된 LF 해시가 없습니다. 수정 커밋 SHA를 사용하세요.")
    if source_git(code, "status", "--porcelain", "--ignore-submodules=all"):
        raise RuntimeError("깨끗한 VLADE 소스 checkout이 필요합니다.")
    return code, ref



import shutil
import sys

SOURCE_REF = meta["source_ref"]
SOURCE_COMMIT = meta["source_commit"]
if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_COMMIT):
    raise ValueError("파일럿 source_commit 형식이 잘못됐습니다.")
source_root = Path("/content/vlade_main_sources")
CODE = source_root / SOURCE_COMMIT
if not CODE.exists():
    source_root.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="restore-", dir=source_root) as stage:
        checkout, actual_commit = prepare_source(SOURCE_REF, Path(stage))
        if actual_commit != SOURCE_COMMIT:
            raise RuntimeError(f"파일럿 Git 커밋 불일치: {actual_commit} != {SOURCE_COMMIT}")
        checkout.rename(CODE)
if source_git(CODE, "rev-parse", "HEAD") != SOURCE_COMMIT:
    raise RuntimeError("본 학습 checkout의 Git 커밋이 파일럿과 다릅니다.")
if source_git(CODE, "status", "--porcelain", "--ignore-submodules=all"):
    raise RuntimeError("본 학습 checkout에 미커밋 변경이 있습니다.")
import importlib

source_root = (CODE / "src").resolve()
for name, module in tuple(sys.modules.items()):
    if name == "vlade" or name.startswith("vlade."):
        module_file = getattr(module, "__file__", None)
        if module_file is not None and not Path(module_file).resolve().is_relative_to(source_root):
            raise RuntimeError(f"이전 VLADE 모듈이 로드돼 있습니다: {name}. Colab 세션을 다시 시작하세요.")

sys.path.insert(0, str(source_root))
importlib.invalidate_caches()

from vlade.training.run_control import verify_checkpoint

WORKSPACE = Path(meta["workspace"])
RUN_ROOT = WORKSPACE / "runs" / PILOT_RUN_GROUP / "suite"
OUTPUT = RUN_ROOT / "checkpoints"
PILOT_ROOT = RUN_ROOT / "pilot4"
PILOT_CHECKPOINT = PILOT_ROOT / "checkpoints" / "step-0000004"
report = json.loads((HANDOFF / "pilot4" / "pilot_report.json").read_text(encoding="utf-8"))
if (
    (report["status"], report["step"], report["checkpoint_verified"]) != ("completed", 4, True)
    or Path(report["checkpoint"]) != PILOT_CHECKPOINT
    or report["fingerprint"]["source_commit"] != SOURCE_COMMIT
    or report["fingerprint"]["suite_map.json"] != meta["suite_map_sha256"]
):
    raise RuntimeError("파일럿 보고서의 완료 상태, 경로, Git 커밋 또는 suite map이 다릅니다.")

CONFIG = CODE / meta["config"]
accumulation = json.loads(CONFIG.read_text(encoding="utf-8"))["train"]["accumulation_steps"]
if not PILOT_ROOT.exists():
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".pilot-", dir=RUN_ROOT) as directory:
        staged = Path(directory) / "pilot4"
        staged_checkpoint = staged / "checkpoints" / PILOT_CHECKPOINT.name
        staged_checkpoint.parent.mkdir(parents=True)
        shutil.copytree(HANDOFF / meta["pilot_checkpoint"], staged_checkpoint)
        if verify_checkpoint(staged_checkpoint, accumulation=accumulation)["step"] != 4:
            raise RuntimeError("파일럿 체크포인트 step 불일치")
        shutil.copy2(HANDOFF / "pilot4" / "pilot_report.json", staged / "pilot_report.json")
        staged.rename(PILOT_ROOT)
else:
    if json.loads((PILOT_ROOT / "pilot_report.json").read_text(encoding="utf-8")) != report:
        raise RuntimeError("SSD에 있는 파일럿 보고서와 Drive 보고서가 다릅니다.")
    if verify_checkpoint(PILOT_CHECKPOINT, accumulation=accumulation)["step"] != 4:
        raise RuntimeError("파일럿 체크포인트 step 불일치")
print("PILOT_RESTORE_OK:", SOURCE_COMMIT)
print(f"로컬 SSD 여유: {shutil.disk_usage(WORKSPACE).free / 1e9:.1f} GB")


In [ ]:
# @title 3. 본 학습 환경 준비 · 최신 체크포인트 확인
import importlib.util
import subprocess

from vlade.research.client import job

LAUNCHER = WORKSPACE / "suite_main_launcher.py"
LAUNCHER.write_text('"""Run the pinned VLADE CLI with the notebook\'s main FFT checkpoint policy."""\nimport importlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\nimport tempfile\n\nSAVE_STEPS = (250, 500, 750, 1000)\nSSD_RETIRE_AT = {750: 250, 1000: 500}\nPAUSED = 75\n\n\nclass ScheduledCheckpointPause(KeyboardInterrupt):\n    pass\n\n\ndef install_policy(module_name, replacements, **bindings):\n    module = importlib.import_module(module_name)\n    path = Path(module.__file__)\n    source = path.read_text(encoding="utf-8")\n    for before, after in replacements:\n        if source.count(before) != 1:\n            raise RuntimeError(f"Pinned VLADE interface changed: {module_name} / {before.strip()}")\n        source = source.replace(before, after)\n    module.__dict__.update(bindings)\n    exec(compile(source, str(path), "exec"), module.__dict__)\n    return module\n\n\ndef copy_checkpoint(source, destination, accumulation, expected_step):\n    from vlade.training.run_control import verify_checkpoint\n\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if destination.exists():\n        if source == destination:\n            seal = verify_checkpoint(destination, accumulation=accumulation)\n        else:\n            seal = verify_checkpoint(source, accumulation=accumulation)\n            if verify_checkpoint(destination, accumulation=accumulation) != seal:\n                raise RuntimeError(f"Checkpoint copy conflicts: {destination}")\n    else:\n        with tempfile.TemporaryDirectory(prefix=".checkpoint-", dir=destination.parent) as directory:\n            staged = Path(directory) / destination.name\n            shutil.copytree(source, staged)\n            seal = verify_checkpoint(staged, accumulation=accumulation)\n            if destination.exists():\n                raise FileExistsError(destination)\n            staged.rename(destination)\n    if seal["step"] != expected_step:\n        raise RuntimeError(f"Unexpected checkpoint step: {destination}")\n    return seal\n\n\ndef copy_status(source, destination):\n    if source.is_file():\n        destination.parent.mkdir(parents=True, exist_ok=True)\n        with tempfile.TemporaryDirectory(prefix=".status-", dir=destination.parent) as directory:\n            staged = Path(directory) / destination.name\n            shutil.copyfile(source, staged)\n            staged.replace(destination)\n\n\ndef install_cli(control_path):\n    return install_policy(\n        "vlade.research.cli",\n        [\n            (\n                \'str(ROOT / "scripts/research/run.py"),\',\n                \'_main_launcher, str(ROOT), _main_control,\',\n            ),\n            (\n                \'config["run_group"] + ".json"\',\n                \'config["run_group"] + "-main.json"\',\n            ),\n            (\n                "        if settings_path.is_file() and json.loads(settings_path.read_text()) != config:",\n                \'        previous = json.loads(settings_path.read_text()) if settings_path.is_file() else config\\n\'\n                \'        if (config["task"], phase, hub.get("enabled")) == ("suite", "main", False):\\n\'\n                \'            previous = {**previous, "hub": {**previous["hub"], "enabled": False}}\\n\'\n                \'        if {**previous, "resume": None} != {**config, "resume": None}:\',\n            ),\n            (\n                \'    if config["task"] in {"fft", "student", "paper", "suite"}:\',\n                \'    main_without_hub = (config["task"] == "suite"\\n\'\n                \'                        and config["settings"].get("phase") == "main"\\n\'\n                \'                        and hub.get("enabled") is False)\\n\'\n                \'    if main_without_hub:\\n\'\n                \'        for name in ("VLADE_HUB_REPO", "VLADE_HUB_RUN_ID"):\\n\'\n                \'            env.pop(name, None)\\n\'\n                \'            os.environ.pop(name, None)\\n\'\n                \'    elif config["task"] in {"fft", "student", "paper", "suite"}:\',\n            ),\n        ],\n        _main_launcher=str(Path(__file__).resolve()), _main_control=str(control_path),\n    )\n\n\ndef prepare_paged_optimizer_resume(optimizer):\n    if not getattr(optimizer, "is_paged", False):\n        return\n    import inspect\n    import textwrap\n    from types import MethodType\n\n    # bitsandbytes 0.43.3\'s default load moves saved paged buffers to ordinary\n    # CUDA storage. Re-create those buffers with the optimizer\'s own allocator.\n    loader = type(optimizer).load_state_dict\n    source = textwrap.dedent(inspect.getsource(loader))\n    changes = (\n        (\n            "    state_dict = deepcopy(state_dict)",\n            \'    state_dict = {**state_dict, "state": {key: dict(value) if isinstance(value, dict) else value\\n\'\n            \'                                        for key, value in state_dict["state"].items()},\\n\'\n            \'                  "param_groups": [dict(group) for group in state_dict["param_groups"]]}\',\n        ),\n        (\n            "                if k in self.non_castable_tensor_keys:",\n            \'                if k in {"state1", "state2"} and param.numel() >= 100000 and v.shape == param.shape:\\n\'\n            \'                    restored = self.get_state_buffer(param, dtype=v.dtype)\\n\'\n            \'                    restored.copy_(v)\\n\'\n            \'                    value[k] = restored\\n\'\n            \'                elif k in self.non_castable_tensor_keys:\',\n        ),\n    )\n    for before, after in changes:\n        if source.count(before) != 1:\n            raise RuntimeError("Unexpected bitsandbytes paged optimizer restore interface.")\n        source = source.replace(before, after)\n    functions = {}\n    exec(compile(source, inspect.getsourcefile(loader), "exec"), loader.__globals__, functions)\n    optimizer.load_state_dict = MethodType(functions["load_state_dict"], optimizer)\n\n\ndef install_worker(control):\n    output = Path(control["output"])\n    drive_output = Path(control["drive_output"])\n    accumulation = control["accumulation"]\n\n    def back_up_checkpoint(path):\n        if path.parent != output or path.name not in {f"step-{step:07d}" for step in SAVE_STEPS}:\n            raise RuntimeError(f"Unexpected FFT checkpoint: {path}")\n        step = int(path.name.removeprefix("step-"))\n        copy_checkpoint(path, drive_output / "checkpoints" / path.name, accumulation, step)\n        print(f"DRIVE_CHECKPOINT_OK: step {step}", flush=True)\n\n        retired_step = SSD_RETIRE_AT.get(step)\n        if retired_step is None:\n            return\n        old_path = output / f"step-{retired_step:07d}"\n        if old_path.is_symlink():\n            raise RuntimeError(f"Refusing to delete a checkpoint symlink: {old_path}")\n        if not old_path.exists():\n            return\n        old_backup = drive_output / "checkpoints" / old_path.name\n        copy_checkpoint(old_path, old_backup, accumulation, retired_step)\n\n        shutil.rmtree(old_path)\n        print(f"SSD_CHECKPOINT_REMOVED: step {retired_step} after verified Drive backup of step {step}", flush=True)\n\n    install_policy(\n        "vlade.training.fft",\n        [\n            (\n                "                if (\\n"\n                "                    budget_stop\\n"\n                "                    or current.step % cfg.save_steps == 0\\n"\n                "                    or current.step == cfg.max_steps\\n"\n                "                    or hub.is_best(metrics)\\n"\n                "                ):",\n                "                if current.step in _main_save_steps:",\n            ),\n            (\n                "                    hub.publish(saved_path, final=current.step == cfg.max_steps)",\n                "                    _back_up_checkpoint(saved_path)",\n            ),\n            (\n                "            state = restore_training_state(payload, optimizer, scheduler)",\n                "            _prepare_paged_optimizer_resume(optimizer)\\n"\n                "            state = restore_training_state(payload, optimizer, scheduler)\\n"\n                "            del payload",\n            ),\n            (\n                \'                raise ValueError("Trainer state step does not match checkpoint metadata")\',\n                \'                raise ValueError("Trainer state step does not match checkpoint metadata")\\n\'\n                \'            print(f"RESUME_STATE_OK: completed={state.step}, next={state.step + 1}, "\\n\'\n                \'                  f"sequence_cursor={state.microsteps * cfg.batch_size}", flush=True)\',\n            ),\n            (\n                "        num_patches = model.vision_backbone.get_num_patches() * cfg.num_images_in_input + 1",\n                "        resume_start_step = state.step\\n"\n                "        num_patches = model.vision_backbone.get_num_patches() * cfg.num_images_in_input + 1",\n            ),\n            (\n                "                if current.step == 1 or current.step % cfg.log_steps == 0:",\n                "                if cfg.resume and current.step == resume_start_step + 1:\\n"\n                \'                    print(f"RESUME_FIRST_STEP_OK: step {current.step}", flush=True)\\n\'\n                "                if current.step == 1 or current.step % cfg.log_steps == 0:",\n            ),\n        ],\n        _main_save_steps=SAVE_STEPS,\n        _back_up_checkpoint=back_up_checkpoint,\n        _prepare_paged_optimizer_resume=prepare_paged_optimizer_resume,\n    )\n    install_policy(\n        "vlade.training.colab_suite_fft",\n        [\n            (\n                \'    if cfg.resume:\\n        if not cfg.resume.resolve().is_relative_to\',\n                \'    if cfg.resume:\\n        cfg = fft.resolve_config_assets(cfg, download=True)\\n\'\n                \'        if not cfg.resume.resolve().is_relative_to\',\n            ),\n            (\n                \'                        save_boundary()\\n\'\n                \'                        raise _SavedBudgetStop("Saved before the next accumulation window")\',\n                \'                        raise _main_pause(f"Budget pause after step {state.step}")\',\n            ),\n            (\n                \'    except BaseException:\\n        with (run_root / "logs/errors.log")\',\n                \'    except _main_pause:\\n        raise\\n\'\n                \'    except BaseException:\\n        with (run_root / "logs/errors.log")\',\n            ),\n        ],\n        _main_pause=ScheduledCheckpointPause,\n    )\n\n\ndef main(args):\n    code, control_path = map(Path, args[:2])\n    cli_args = args[2:]\n    sys.path[:0] = [str(code / "src"), str(code)]\n    control = json.loads(control_path.read_text(encoding="utf-8"))\n    cli = install_cli(control_path)\n    worker = "--worker" in cli_args\n    if worker:\n        install_worker(control)\n    try:\n        return cli.main(cli_args)\n    except ScheduledCheckpointPause as error:\n        if not worker:\n            raise\n        print(str(error), flush=True)\n        return PAUSED\n    except cli.subprocess.CalledProcessError as error:\n        if worker or error.returncode != PAUSED:\n            raise\n        return PAUSED\n\n\nif __name__ == "__main__":\n    raise SystemExit(main(sys.argv[1:]))\n', encoding="utf-8")
spec = importlib.util.spec_from_file_location("vlade_suite_main_launcher", LAUNCHER)
orchestration = importlib.util.module_from_spec(spec)
spec.loader.exec_module(orchestration)

SAVE_STEPS = orchestration.SAVE_STEPS
DRIVE_CHECKPOINTS = DRIVE_OUTPUT / "checkpoints"
control_path = WORKSPACE / f"{PILOT_RUN_GROUP}-main-control.json"
control_path.write_text(json.dumps({
    "output": str(OUTPUT), "drive_output": str(DRIVE_OUTPUT), "accumulation": accumulation
}), encoding="utf-8")

main = dict(meta["main_overrides"])
if main["run_group"] != PILOT_RUN_GROUP or (
    main["train.sequence_length"], main["train.sequence_sampling"],
    main["train.detach_every"], main["train.temporal_feature_layer"]
) != (4, "episode_uniform", 2, -2):
    raise RuntimeError("파일럿의 Suite-balanced K4 학습 설정과 일치하지 않습니다.")
# Keep the pinned config value compatible with any existing 500-step run.
# The launcher enforces the explicit SAVE_STEPS schedule.
main.update({
    "settings.phase": "main", "train.max_steps": 1000, "train.save_steps": 500,
    "budget.release_colab_on_deadline": False, "hub.enabled": False,
})
for folder in (OUTPUT, DRIVE_CHECKPOINTS):
    unexpected = [p.name for p in folder.glob("step-*")
                  if p.name not in {f"step-{step:07d}" for step in SAVE_STEPS}]
    if unexpected:
        raise RuntimeError(f"이 실행 그룹의 저장 step 설정이 다릅니다: {unexpected}")

def check_main_checkpoint(path, step):
    if verify_checkpoint(path, accumulation=accumulation)["step"] != step:
        raise RuntimeError("본 학습 체크포인트 step 불일치")
    details = json.loads((path / "vlade_fft.json").read_text(encoding="utf-8"))
    if (
        details["source_revision"] != SOURCE_COMMIT
        or details["configuration"]["max_steps"] != 1000
        or details["configuration"]["save_steps"] != 500
        or details["configuration"]["sequence_length"] != 4
        or details["training_code_sha256"]["suite_map.json"] != meta["suite_map_sha256"]
    ):
        raise RuntimeError("본 학습 체크포인트의 소스 또는 실행 설정이 다릅니다.")
    if details["configuration"]["output_dir"] != str(OUTPUT):
        raise RuntimeError("다른 본 학습 실행의 체크포인트입니다.")


def available_checkpoint(step):
    name = f"step-{step:07d}"
    local, backed_up = OUTPUT / name, DRIVE_CHECKPOINTS / name
    if local.exists():
        check_main_checkpoint(local, step)
        return local
    if backed_up.exists():
        check_main_checkpoint(backed_up, step)
        return backed_up
    return None

def latest_resume():
    for step in reversed(SAVE_STEPS[:-1]):
        checkpoint = available_checkpoint(step)
        if checkpoint is not None:
            return step, checkpoint
    return None, None

finished = available_checkpoint(1000)
resume_step, resume = (None, None) if finished else latest_resume()
print("저장 step:", list(SAVE_STEPS))
main.pop("resume", None)
if finished is not None:
    print("MAIN_CHECKPOINT_EXISTS: step 1000", finished)
else:
    if resume is not None:
        resume_local = OUTPUT / resume.name
        if resume != resume_local:
            orchestration.copy_checkpoint(resume, resume_local, accumulation, resume_step)
        main["resume"] = str(resume_local)
        print(f"RESUME_READY: step {resume_step}")
    else:
        print("START_READY: T0")
    job(CONFIG, action="plan", workspace=WORKSPACE, overrides=main)
    # Keep the main session budget separate from any pilot session in the same workspace.
    launcher_args = [
        sys.executable, str(LAUNCHER), str(CODE), str(control_path),
        "--config", str(CONFIG), "--workspace", str(WORKSPACE),
    ]
    for key, value in main.items():
        launcher_args += ["--set", f"{key}={json.dumps(value)}"]
    subprocess.run([*launcher_args, "--action", "setup"], check=True)
    print("SETUP_OK")


In [ ]:
# @title 4. 본 학습 실행 · Drive 상태 기록
RUN_MAIN = True # @param {type:"boolean"}
SEND_GMAIL = False # @param {type:"boolean"}
UNASSIGN_ON_COMPLETE = False # @param {type:"boolean"}

def notify_gmail(status):
    if not SEND_GMAIL:
        return
    import smtplib
    import ssl
    from email.message import EmailMessage

    try:
        address = (userdata.get("GMAIL_ADDRESS") or "").strip()
        app_password = "".join((userdata.get("GMAIL_APP_PASSWORD") or "").split())
        if "@" not in address or len(app_password) != 16:
            raise ValueError("Gmail Secrets")
        message = EmailMessage()
        message["From"] = address
        message["To"] = address
        message["Subject"] = f"VLADE K4 FFT {status['status']}"
        message.set_content(json.dumps(status, indent=2, ensure_ascii=False))
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20,
                              context=ssl.create_default_context()) as smtp:
            smtp.login(address, app_password)
            smtp.send_message(message)
    except Exception as error:
        print(f"Gmail 알림 실패 ({type(error).__name__})")

def run_main():
    drive.mount("/content/drive")
    DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
    result = None
    failure = None
    already_finished = available_checkpoint(1000)
    if already_finished is None:
        latest_step, latest = latest_resume()
        if latest is not None and main.get("resume") != str(OUTPUT / latest.name):
            raise RuntimeError(f"{latest_step}-step 체크포인트가 생겼습니다. 3번 셀로 재개 경로를 먼저 준비하세요.")
    try:
        if already_finished is None:
            subprocess.run(["nvidia-smi", "-L"], check=True)
            result = subprocess.run([*launcher_args, "--action", "run"]).returncode
    except (Exception, KeyboardInterrupt) as error:
        failure = error
    for step in SAVE_STEPS:
        name = f"step-{step:07d}"
        path = OUTPUT / name
        if (path / "COMPLETE").is_file() and not (DRIVE_CHECKPOINTS / name).exists():
            orchestration.copy_checkpoint(path, DRIVE_CHECKPOINTS / name, accumulation, step)
    for folder in (RUN_ROOT / "logs", RUN_ROOT / "reports"):
        if folder.is_dir():
            shutil.copytree(folder, DRIVE_OUTPUT / folder.name, dirs_exist_ok=True)
    for pattern in ("run-*.json", "metrics-*.jsonl"):
        for path in OUTPUT.glob(pattern):
            orchestration.copy_status(path, DRIVE_OUTPUT / "logs" / path.name)

    saved = [step for step in SAVE_STEPS
             if (DRIVE_CHECKPOINTS / f"step-{step:07d}" / "COMPLETE").is_file()]
    if failure is not None or result not in (None, 0, orchestration.PAUSED):
        label = "failed"
    elif result == orchestration.PAUSED:
        label = "budget_paused"
    elif 1000 in saved:
        label = "completed" if result == 0 else "checkpoint_already_saved"
    else:
        label = "failed"
    status = {
        "status": label, "run_group": PILOT_RUN_GROUP, "source_commit": SOURCE_COMMIT,
        "scheduled_steps": list(SAVE_STEPS), "drive_saved_steps": saved,
        "ssd_saved_steps": [
            step for step in SAVE_STEPS if (OUTPUT / f"step-{step:07d}" / "COMPLETE").is_file()
        ],
        "hub_upload_enabled": False,
        "pc_export_source": str(DRIVE_CHECKPOINTS / "step-0001000") if 1000 in saved else None,
        "return_code": result,
    }
    (DRIVE_OUTPUT / "status.json").write_text(json.dumps(status, indent=2) + "\n", encoding="utf-8")
    drive.flush_and_unmount(timeout_ms=1_800_000)
    print(json.dumps(status, indent=2, ensure_ascii=False))
    notify_gmail(status)
    if failure is not None:
        raise failure
    if label == "failed":
        raise RuntimeError("본 학습이 실패했습니다. 실행 출력과 Drive의 main_fft/logs를 확인하세요.")
    if UNASSIGN_ON_COMPLETE and label in {"completed", "checkpoint_already_saved"}:
        from google.colab import runtime
        runtime.unassign()

if RUN_MAIN:
    run_main()
else:
    print("본 학습을 시작하려면 RUN_MAIN을 켜고 이 셀을 실행하세요.")


## 1,000-step 체크포인트를 Windows PC에 저장

1. 4번 셀에서 Drive 저장이 끝나면 5번 셀의 **DOWNLOAD_PC_SCRIPT**를 켜서 복사 스크립트를 받으세요.
2. PC의 Google Drive 데스크톱 앱에서 `main_fft/checkpoints/step-0001000` 폴더를 찾으세요. 웹 Drive에서 이미 내려받아 압축을 풀었다면 그 폴더를 원본으로 지정해도 됩니다.
3. PowerShell에서 Python 3.11 이상으로 아래 명령을 실행하세요. Google Drive 드라이브 문자와 폴더 이름은 PC에 표시된 경로에 맞춰 바꾸세요. 로컬 저장 공간은 파일럿 실측 기준 약 30.7 GB가 필요합니다.

```powershell
python "$env:USERPROFILE\Downloads\copy_vlade_step_1000_to_pc.py" --source "G:\My Drive\VLADE\runs\ava_fft_k4\fish13031e-ava-suite-k4-seed42-20261111T062518Z-2f8a601c\main_fft\checkpoints\step-0001000" --destination "$env:USERPROFILE\Downloads\VLADE\fish13031e-ava-suite-k4-seed42-20261111T062518Z-2f8a601c\step-0001000"
```

스크립트는 파일별 크기와 SHA-256을 확인합니다. `PC_CHECKPOINT_OK`가 뜨면, 나중에 Hugging Face에 직접 올릴 때 PC의 `step-0001000` 폴더 전체를 사용하세요.


In [ ]:
# @title 5. 1,000-step 체크포인트 PC 복사 스크립트
DOWNLOAD_PC_SCRIPT = False # @param {type:"boolean"}

if DOWNLOAD_PC_SCRIPT:
    from google.colab import files

    drive.mount("/content/drive")
    pc_source = DRIVE_CHECKPOINTS / "step-0001000"
    if not (pc_source / "COMPLETE").is_file():
        raise RuntimeError("Drive에 봉인된 1,000-step 체크포인트가 있어야 합니다.")
    pc_manifest = json.loads((pc_source / "checkpoint_manifest.json").read_text(encoding="utf-8"))
    if pc_manifest.get("step") != 1000:
        raise RuntimeError("1,000-step 체크포인트 봉인이 일치하지 않습니다.")
    pc_script = Path("/content/copy_vlade_step_1000_to_pc.py")
    pc_script.write_text('"""Copy the sealed VLADE step-1000 checkpoint from Google Drive to a local disk."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport tempfile\n\nMANIFEST = "checkpoint_manifest.json"\nCOMPLETE = "COMPLETE"\n\n\ndef file_hash(path):\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef matches(path, expected):\n    return (\n        path.is_file() and not path.is_symlink()\n        and path.stat().st_size == expected["bytes"]\n        and file_hash(path) == expected["sha256"]\n    )\n\n\ndef copy_checkpoint(source, destination):\n    source = source.resolve(strict=True)\n    destination = destination.resolve()\n    if destination == source or destination.is_relative_to(source):\n        raise ValueError("Choose a separate local PC folder for the destination.")\n    if source.name != "step-0001000" or not (source / COMPLETE).is_file():\n        raise ValueError("Select the completed step-0001000 folder from Drive.")\n\n    manifest_bytes = (source / MANIFEST).read_bytes()\n    manifest = json.loads(manifest_bytes)\n    if (manifest.get("schema_version"), manifest.get("step")) != (1, 1000):\n        raise ValueError("Expected a step-1000 checkpoint seal.")\n    payload = manifest["files"]\n    if not payload:\n        raise ValueError("Checkpoint seal has no payload files.")\n    for name in payload:\n        relative = PurePosixPath(name)\n        if relative.is_absolute() or ".." in relative.parts or "\\\\" in name or ":" in name:\n            raise ValueError(f"Unexpected checkpoint path: {name}")\n        if name in {MANIFEST, COMPLETE} or relative.as_posix() != name:\n            raise ValueError(f"Unexpected checkpoint path: {name}")\n\n    destination.mkdir(parents=True, exist_ok=True)\n    allowed = {*payload, MANIFEST, COMPLETE}\n    existing = {\n        path.relative_to(destination).as_posix() for path in destination.rglob("*")\n        if path.is_file() or path.is_symlink()\n    }\n    if existing - allowed:\n        raise ValueError("The destination contains unrelated files. Choose a dedicated folder.")\n    completed = (destination / COMPLETE).exists()\n    if (destination / MANIFEST).is_file() and (destination / MANIFEST).read_bytes() != manifest_bytes:\n        raise ValueError("The destination contains another checkpoint. Choose a different folder.")\n    if completed and existing != allowed:\n        raise ValueError("A completed destination is missing files. Choose a different folder.")\n\n    for name, expected in payload.items():\n        target = destination / name\n        if not target.resolve().is_relative_to(destination):\n            raise ValueError(f"Unexpected local path: {name}")\n        if matches(target, expected):\n            continue\n        if completed:\n            raise ValueError(f"Existing completed copy failed verification: {name}")\n        original = source / name\n        if not original.resolve().is_relative_to(source) or not original.is_file():\n            raise ValueError(f"Invalid Drive checkpoint file: {name}")\n        target.parent.mkdir(parents=True, exist_ok=True)\n        with tempfile.TemporaryDirectory(prefix=".vlade-copy-", dir=destination.parent) as directory:\n            staged = Path(directory) / "file"\n            shutil.copyfile(original, staged)\n            if not matches(staged, expected):\n                raise ValueError(f"Copied file failed SHA-256 or size verification: {name}")\n            staged.replace(target)\n        print(f"Verified {name}", flush=True)\n\n    if not completed:\n        with tempfile.TemporaryDirectory(prefix=".vlade-seal-", dir=destination.parent) as directory:\n            stage = Path(directory)\n            (stage / MANIFEST).write_bytes(manifest_bytes)\n            shutil.copyfile(source / COMPLETE, stage / COMPLETE)\n            (stage / MANIFEST).replace(destination / MANIFEST)\n            (stage / COMPLETE).replace(destination / COMPLETE)\n    print(f"PC_CHECKPOINT_OK: {destination}")\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--source", required=True, type=Path)\n    parser.add_argument("--destination", required=True, type=Path)\n    options = parser.parse_args()\n    copy_checkpoint(options.source, options.destination)\n', encoding="utf-8")
    print("PC로 복사할 Drive 폴더:", pc_source)
    print(f"체크포인트 파일 용량: {sum(x['bytes'] for x in pc_manifest['files'].values()) / 1e9:.1f} GB")
    files.download(str(pc_script))
else:
    print("Drive 저장이 완료되면 DOWNLOAD_PC_SCRIPT를 켜고 실행하세요.")
